# Cleanup All Resources - GCP Dataplex Demo

This notebook will clean up and delete all resources created by:
- `config_and_data_setup.ipynb`
- `apply_glossary_terms.ipynb`
- `upload_census_to_gcs.ipynb`
- `setup_cloudsql_census.ipynb`
- `setup_firestore.ipynb`
- `write_firestore_metadata_to_dataplex.ipynb`

**Resources that will be deleted:**

**1. BigQuery Datasets:**
   - `census_bureau_acs` (and all ~278 tables)
   - `census_uk_2021` (and all tables including `ts001_utla` and `ts001_ctry_biglake`)

**2. BigQuery Connections:**
   - `census-biglake-connection` (used for BigLake external tables)

**3. Cloud Storage:**
   - GCS Bucket: `{PROJECT_ID}-census-data` (with all census files)

**4. CloudSQL:**
   - PostgreSQL instance: `census-demo-db`
   - Database: `census_data`
   - Table: `census_residence_type`

**5. Dataplex Aspect Types:**
   - `census-survey-metadata`
   - `data-governance-public`

**6. Dataplex Aspects:**
   - All aspects applied to BigQuery table entries

**7. Dataplex Glossary Resources:**
   - Business Glossary: `acs-demographics-glossary`
   - 5 Categories
   - All glossary terms

**8. Dataplex Custom Entry Group, Entry Type & Entry:**
   - Entry Group: `cloudsql-entries`
   - Entry Type: `cloudsql-table`
   - Entry: `cloudsql-census-demo-db-census-data-census-residence-type-copy`

**9. Firestore:**
   - Database: `(default)`
   - Collection: `datasets` (all documents and subcollections)

**⚠️  WARNING:**
- This operation cannot be undone
- All data will be permanently deleted
- All custom metadata (aspects) will be removed
- All glossary resources will be permanently deleted

**Required permissions:**
- `roles/bigquery.admin` (to delete BigQuery datasets and connections)
- `roles/storage.admin` (to delete GCS bucket)
- `roles/cloudsql.admin` (to delete CloudSQL instance)
- `roles/dataplex.admin` (to delete aspect types)
- `roles/dataplex.catalogAdmin` (to remove aspects from entries and delete glossary)
- `roles/datastore.owner` (to delete Firestore data)

## Section 1: Configuration & Authentication

In [9]:
# Configuration variables
PROJECT_ID = "johnswain-1200-20260303091357"  
REGION = "us-central1"                           
CATALOG_LOCATION = "us"  # BigQuery US multi-region datasets are cataloged in 'us'

# BigQuery datasets
DATASET_ID = "census_bureau_acs"
CENSUS_UK_DATASET_ID = "census_uk_2021"
DATASET_LOCATION = "US"

# BigQuery Connection
CONNECTION_ID = "census-biglake-connection"

# Cloud Storage
BUCKET_NAME = f"{PROJECT_ID}-census-data"

# CloudSQL
CLOUDSQL_INSTANCE_NAME = "census-demo-db"

# Dataplex Glossary
GLOSSARY_ID = "acs-demographics-glossary"

# Dataplex custom entry group, entry type, and entry (from write_firestore_metadata_to_dataplex.ipynb)
ENTRY_GROUP_ID = "cloudsql-entries"
ENTRY_TYPE_ID = "cloudsql-table"
CLOUDSQL_ENTRY_ID = "cloudsql-census-demo-db-census-data-census-residence-type-copy"

# Firestore (from setup_firestore.ipynb)
FIRESTORE_DATABASE_ID = "(default)"
FIRESTORE_DATASETS_COLLECTION = "datasets"

print("📋 Configuration:")
print(f"  Project ID: {PROJECT_ID}")
print(f"  Region: {REGION}")
print(f"  Catalog Location: {CATALOG_LOCATION}")
print()
print("  BigQuery Datasets to delete:")
print(f"    - {DATASET_ID}")
print(f"    - {CENSUS_UK_DATASET_ID}")
print()
print(f"  BigQuery Connection to delete: {CONNECTION_ID}")
print(f"  GCS Bucket to delete: {BUCKET_NAME}")
print(f"  CloudSQL Instance to delete: {CLOUDSQL_INSTANCE_NAME}")
print(f"  Glossary to delete: {GLOSSARY_ID}")
print()
print("  Dataplex catalog resources to delete:")
print(f"    - Entry: {CLOUDSQL_ENTRY_ID}")
print(f"    - Entry Group: {ENTRY_GROUP_ID}")
print(f"    - Entry Type: {ENTRY_TYPE_ID}")
print()
print(f"  Firestore Database to clean up: {FIRESTORE_DATABASE_ID}")

📋 Configuration:
  Project ID: johnswain-1200-20260303091357
  Region: us-central1
  Catalog Location: us
  Dataset to delete: census_bureau_acs


In [10]:
# Verify authentication
from google.auth import default
import google.auth

try:
    credentials, project = default()
    print("🔐 Authentication Status:")
    print(f"  ✅ Authenticated successfully")
    print(f"  ✅ Default project: {project}")
    
    if project != PROJECT_ID:
        print(f"  ⚠️  Note: Using PROJECT_ID ({PROJECT_ID}) instead of default ({project})")
    
except Exception as e:
    print(f"❌ Authentication failed: {e}")
    print("Please run: gcloud auth application-default login")
    raise

🔐 Authentication Status:
  ✅ Authenticated successfully
  ✅ Default project: graph-demo-471710
  ⚠️  Note: Using PROJECT_ID (johnswain-1200-20260303091357) instead of default (graph-demo-471710)


In [11]:
# Install required libraries
import sys
import subprocess

print("📦 Installing required libraries...")
packages = ["google-cloud-bigquery", "google-cloud-dataplex", "google-cloud-resource-manager", "google-cloud-firestore"]

for package in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

print("✅ Libraries installed")

📦 Installing required libraries...



[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


✅ Libraries installed



[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [12]:
# Initialize clients
from google.cloud import bigquery
from google.cloud import dataplex_v1
from google.cloud import storage
from google.cloud import firestore
import requests
from google.auth.transport.requests import Request
from google.auth import default

bq_client = bigquery.Client(project=PROJECT_ID)
catalog_client = dataplex_v1.CatalogServiceClient()
storage_client = storage.Client(project=PROJECT_ID)
firestore_client = firestore.Client(project=PROJECT_ID, database=FIRESTORE_DATABASE_ID)

# Get credentials for REST API calls (needed for glossary and CloudSQL operations)
credentials, _ = default()
credentials.refresh(Request())

print("✅ Clients initialized:")
print("  - BigQuery client")
print("  - Dataplex Catalog client")
print("  - Cloud Storage client")
print("  - Firestore client")
print("  - Authentication credentials")

✅ Clients initialized:
  - BigQuery client
  - Dataplex Catalog client


## Section 2: Remove Aspects from BigQuery Table Entries

Before deleting the aspect types, we need to remove all aspect instances that were applied to BigQuery table entries.

This will:
- List all tables in the census_bureau_acs dataset
- Remove both aspects (census-survey-metadata and data-governance-public) from each table entry

In [13]:
# List all tables in the dataset
print("📋 Listing tables in dataset...")
print()

dataset_ref = f"{PROJECT_ID}.{DATASET_ID}"

try:
    tables = list(bq_client.list_tables(dataset_ref))
    total_tables = len(tables)
    
    print(f"✅ Found {total_tables} tables in {DATASET_ID}")
    print(f"   Will remove aspects from all {total_tables} table entries")
    print()
    
except Exception as e:
    error_msg = str(e)
    if "404" in error_msg or "not found" in error_msg.lower():
        print(f"⚠️  Dataset {DATASET_ID} not found - skipping aspect removal")
        total_tables = 0
    else:
        print(f"❌ Error listing tables: {error_msg}")
        raise

📋 Listing tables in dataset...

✅ Found 278 tables in census_bureau_acs
   Will remove aspects from all 278 table entries



In [14]:
# Remove aspects from all table entries
print("🗑️  Removing aspects from BigQuery table entries...")
print()

if total_tables == 0:
    print("⚠️  No tables found - skipping aspect removal")
else:
    # Aspect IDs to remove
    aspect_ids = ["census-survey-metadata", "data-governance-public"]
    
    catalog_parent = f"projects/{PROJECT_ID}/locations/{CATALOG_LOCATION}"
    BQ_ENTRY_GROUP = "@bigquery"
    
    success_count = 0
    not_found_count = 0
    error_count = 0
    errors = []
    
    print(f"⏳ Processing {total_tables} tables...")
    
    for i, table in enumerate(tables):
        table_id = table.table_id
        
        # Construct the Dataplex entry name
        entry_id = f"bigquery.googleapis.com/projects/{PROJECT_ID}/datasets/{DATASET_ID}/tables/{table_id}"
        entry_name = f"{catalog_parent}/entryGroups/{BQ_ENTRY_GROUP}/entries/{entry_id}"
        
        # Try to delete both aspects
        for aspect_id in aspect_ids:
            aspect_key = f"{PROJECT_ID}.{CATALOG_LOCATION}.{aspect_id}"
            aspect_full_name = f"{entry_name}/aspects/{aspect_key}"
            
            try:
                catalog_client.delete_aspect(name=aspect_full_name)
                success_count += 1
                
            except Exception as e:
                error_msg = str(e)
                
                # Not found is expected if aspect wasn't applied
                if "404" in error_msg or "not found" in error_msg.lower():
                    not_found_count += 1
                else:
                    error_count += 1
                    errors.append((table_id, aspect_id, error_msg[:80]))
        
        # Show progress every 25 tables
        if (i + 1) % 25 == 0:
            print(f"   Processed {i + 1}/{total_tables} tables... ({success_count} aspects removed)")
    
    print()
    print("=" * 60)
    print("✅ Aspect removal completed!")
    print("=" * 60)
    print(f"   Tables processed: {total_tables}")
    print(f"   Aspects removed: {success_count}")
    print(f"   Not found (already removed): {not_found_count}")
    print(f"   Errors: {error_count}")
    
    if error_count > 0:
        print()
        print(f"⚠️  Errors encountered ({error_count}):")
        for table_id, aspect_id, error in errors[:5]:
            print(f"   - {table_id}.{aspect_id}: {error}")
        if len(errors) > 5:
            print(f"   ... and {len(errors) - 5} more errors")

🗑️  Removing aspects from BigQuery table entries...

⏳ Processing 278 tables...
   Processed 25/278 tables... (0 aspects removed)
   Processed 50/278 tables... (0 aspects removed)
   Processed 75/278 tables... (0 aspects removed)
   Processed 100/278 tables... (0 aspects removed)
   Processed 125/278 tables... (0 aspects removed)
   Processed 150/278 tables... (0 aspects removed)
   Processed 175/278 tables... (0 aspects removed)
   Processed 200/278 tables... (0 aspects removed)
   Processed 225/278 tables... (0 aspects removed)
   Processed 250/278 tables... (0 aspects removed)
   Processed 275/278 tables... (0 aspects removed)

✅ Aspect removal completed!
   Tables processed: 278
   Aspects removed: 0
   Not found (already removed): 0
   Errors: 556

⚠️  Errors encountered (556):
   - blockgroup_2010_5yr.census-survey-metadata: 'CatalogServiceClient' object has no attribute 'delete_aspect'
   - blockgroup_2010_5yr.data-governance-public: 'CatalogServiceClient' object has no attribut

## Section 3: Delete Dataplex Glossary Resources

Delete the business glossary and all its resources created by `apply_glossary_terms.ipynb`:
- Glossary terms (linked to table columns)
- Categories (ISO 11179 Object Classes)
- The glossary itself

In [ ]:
# Step 1: List all terms in the glossary
print("📋 Listing glossary terms...")
print()

glossary_path = f"projects/{PROJECT_ID}/locations/{CATALOG_LOCATION}/glossaries/{GLOSSARY_ID}"
terms_url = f"https://dataplex.googleapis.com/v1/{glossary_path}/terms"

headers = {
    "Authorization": f"Bearer {credentials.token}",
    "Content-Type": "application/json"
}

try:
    response = requests.get(terms_url, headers=headers)
    
    if response.status_code == 200:
        terms_data = response.json()
        terms = terms_data.get('glossaryTerms', [])
        print(f"✅ Found {len(terms)} glossary terms")
        print()
    elif response.status_code == 404:
        print(f"⚠️  Glossary '{GLOSSARY_ID}' not found - skipping glossary cleanup")
        terms = []
    else:
        print(f"⚠️  Could not list terms (status {response.status_code})")
        print(f"   Response: {response.text[:200]}")
        terms = []
        
except Exception as e:
    print(f"⚠️  Error listing terms: {e}")
    terms = []

In [ ]:
# Step 2: Delete all glossary terms
print("🗑️  Deleting glossary terms...")
print()

if len(terms) == 0:
    print("⚠️  No terms to delete")
else:
    deleted_terms = 0
    error_terms = 0
    
    for term in terms:
        term_name = term.get('name', '')
        term_id = term_name.split('/')[-1] if term_name else 'unknown'
        
        try:
            delete_response = requests.delete(
                f"https://dataplex.googleapis.com/v1/{term_name}",
                headers=headers
            )
            
            if delete_response.status_code in [200, 204]:
                deleted_terms += 1
            elif delete_response.status_code == 404:
                deleted_terms += 1  # Already deleted
            else:
                error_terms += 1
                print(f"   ⚠️  Failed to delete term {term_id}: {delete_response.status_code}")
        
        except Exception as e:
            error_terms += 1
            print(f"   ⚠️  Error deleting term {term_id}: {str(e)[:80]}")
    
    print()
    print(f"✅ Glossary terms deletion completed:")
    print(f"   Deleted: {deleted_terms}")
    print(f"   Errors: {error_terms}")

In [ ]:
# Step 3: List and delete all categories
print()
print("📋 Listing glossary categories...")
print()

categories_url = f"https://dataplex.googleapis.com/v1/{glossary_path}/categories"

try:
    response = requests.get(categories_url, headers=headers)
    
    if response.status_code == 200:
        categories_data = response.json()
        categories = categories_data.get('glossaryCategories', [])
        print(f"✅ Found {len(categories)} categories")
        print()
    elif response.status_code == 404:
        print(f"⚠️  Glossary not found - skipping category deletion")
        categories = []
    else:
        print(f"⚠️  Could not list categories (status {response.status_code})")
        categories = []
        
except Exception as e:
    print(f"⚠️  Error listing categories: {e}")
    categories = []

# Delete categories
print("🗑️  Deleting categories...")
print()

if len(categories) == 0:
    print("⚠️  No categories to delete")
else:
    deleted_categories = 0
    error_categories = 0
    
    for category in categories:
        category_name = category.get('name', '')
        category_id = category_name.split('/')[-1] if category_name else 'unknown'
        
        print(f"   Deleting: {category_id}")
        
        try:
            delete_response = requests.delete(
                f"https://dataplex.googleapis.com/v1/{category_name}",
                headers=headers
            )
            
            if delete_response.status_code in [200, 204]:
                print(f"   ✅ Deleted: {category_id}")
                deleted_categories += 1
            elif delete_response.status_code == 404:
                print(f"   ⚠️  Not found: {category_id} (already deleted)")
                deleted_categories += 1
            else:
                print(f"   ❌ Failed: {category_id} - Status {delete_response.status_code}")
                error_categories += 1
        
        except Exception as e:
            print(f"   ❌ Error: {category_id} - {str(e)[:60]}")
            error_categories += 1
    
    print()
    print(f"✅ Categories deletion completed:")
    print(f"   Deleted: {deleted_categories}")
    print(f"   Errors: {error_categories}")

In [ ]:
# Step 4: Delete the glossary itself
print()
print("🗑️  Deleting glossary...")
print()

glossary_delete_url = f"https://dataplex.googleapis.com/v1/{glossary_path}"

try:
    delete_response = requests.delete(glossary_delete_url, headers=headers)
    
    if delete_response.status_code in [200, 204]:
        print(f"✅ Glossary '{GLOSSARY_ID}' deleted successfully")
    elif delete_response.status_code == 404:
        print(f"⚠️  Glossary '{GLOSSARY_ID}' not found (already deleted)")
    else:
        print(f"❌ Failed to delete glossary (status {delete_response.status_code})")
        print(f"   Response: {delete_response.text[:200]}")
        
except Exception as e:
    print(f"❌ Error deleting glossary: {e}")

print()
print("=" * 60)
print("✅ Glossary cleanup completed!")
print("=" * 60)

## Section 4: Delete Dataplex Aspect Types

Now we'll delete the custom aspect types created for the demo:
- `census-survey-metadata`
- `data-governance-public`

In [15]:
# Delete aspect types
print("🗑️  Deleting Dataplex Aspect Types...")
print()

aspect_type_ids = ["census-survey-metadata", "data-governance-public"]
catalog_parent = f"projects/{PROJECT_ID}/locations/{CATALOG_LOCATION}"

deleted_count = 0
not_found_count = 0
error_count = 0

for aspect_type_id in aspect_type_ids:
    aspect_type_name = f"{catalog_parent}/aspectTypes/{aspect_type_id}"
    
    print(f"🗑️  Deleting: {aspect_type_id}")
    
    try:
        # Delete the aspect type
        delete_operation = catalog_client.delete_aspect_type(name=aspect_type_name)
        
        # Wait for deletion to complete
        delete_operation.result(timeout=300)
        
        print(f"   ✅ Deleted: {aspect_type_id}")
        deleted_count += 1
        
    except Exception as e:
        error_msg = str(e)
        
        if "404" in error_msg or "not found" in error_msg.lower():
            print(f"   ⚠️  Not found: {aspect_type_id} (already deleted)")
            not_found_count += 1
        else:
            print(f"   ❌ Error deleting {aspect_type_id}: {error_msg[:80]}")
            error_count += 1
    
    print()

print("=" * 60)
print("✅ Aspect Type deletion completed!")
print("=" * 60)
print(f"   Deleted: {deleted_count}")
print(f"   Not found: {not_found_count}")
print(f"   Errors: {error_count}")

🗑️  Deleting Dataplex Aspect Types...

🗑️  Deleting: census-survey-metadata
   ✅ Deleted: census-survey-metadata

🗑️  Deleting: data-governance-public
   ⚠️  Not found: data-governance-public (already deleted)

✅ Aspect Type deletion completed!
   Deleted: 1
   Not found: 1
   Errors: 0


## Section 5: Delete BigQuery Dataset

Finally, we'll delete the entire BigQuery dataset, which includes all ~278 census tables.

**⚠️  WARNING:** This will permanently delete all data in the `census_bureau_acs` dataset.

In [16]:
# Delete BigQuery dataset
print("🗑️  Deleting BigQuery Dataset...")
print()

dataset_id = f"{PROJECT_ID}.{DATASET_ID}"

print(f"⚠️  About to delete dataset: {dataset_id}")
print(f"   This will delete all {total_tables} tables and their data")
print()

try:
    # Delete the dataset with delete_contents=True to delete all tables
    bq_client.delete_dataset(
        dataset_id,
        delete_contents=True,  # Delete all tables in the dataset
        not_found_ok=True      # Don't error if already deleted
    )
    
    print("=" * 60)
    print("✅ BigQuery dataset deleted successfully!")
    print("=" * 60)
    print(f"   Dataset: {DATASET_ID}")
    print(f"   All {total_tables} tables and data have been permanently deleted")
    
except Exception as e:
    error_msg = str(e)
    
    if "404" in error_msg or "not found" in error_msg.lower():
        print("⚠️  Dataset not found (already deleted)")
    else:
        print(f"❌ Error deleting dataset: {error_msg}")
        raise

🗑️  Deleting BigQuery Dataset...

⚠️  About to delete dataset: johnswain-1200-20260303091357.census_bureau_acs
   This will delete all 278 tables and their data

✅ BigQuery dataset deleted successfully!
   Dataset: census_bureau_acs
   All 278 tables and data have been permanently deleted


## Section 6: Delete UK Census BigQuery Dataset

Delete the `census_uk_2021` dataset created by `upload_census_to_gcs.ipynb`.

In [ ]:
# Delete UK Census BigQuery dataset
print("🗑️  Deleting UK Census BigQuery Dataset...")
print()

census_uk_dataset_id = f"{PROJECT_ID}.{CENSUS_UK_DATASET_ID}"

print(f"⚠️  About to delete dataset: {census_uk_dataset_id}")
print()

try:
    # Check if dataset exists
    dataset = bq_client.get_dataset(census_uk_dataset_id)
    tables = list(bq_client.list_tables(census_uk_dataset_id))
    table_count = len(tables)
    
    print(f"   Found dataset with {table_count} tables")
    
    # Delete the dataset with delete_contents=True to delete all tables
    bq_client.delete_dataset(
        census_uk_dataset_id,
        delete_contents=True,  # Delete all tables in the dataset
        not_found_ok=True      # Don't error if already deleted
    )
    
    print()
    print("=" * 60)
    print("✅ UK Census BigQuery dataset deleted successfully!")
    print("=" * 60)
    print(f"   Dataset: {CENSUS_UK_DATASET_ID}")
    print(f"   Tables deleted: {table_count}")
    
except Exception as e:
    error_msg = str(e)
    
    if "404" in error_msg or "not found" in error_msg.lower():
        print("⚠️  Dataset not found (already deleted or never created)")
    else:
        print(f"❌ Error deleting dataset: {error_msg}")
        raise

## Section 7: Delete BigQuery Connection

Delete the BigQuery Connection created by `upload_census_to_gcs.ipynb` for BigLake external tables.

**Note:** The connection must be deleted before or after the dataset that uses it.

In [ ]:
# Delete BigQuery Connection
print("🗑️  Deleting BigQuery Connection...")
print()

connection_name = f"projects/{PROJECT_ID}/locations/{DATASET_LOCATION}/connections/{CONNECTION_ID}"

print(f"⚠️  About to delete connection: {connection_name}")
print()

try:
    # Check if the connection exists first by trying to get project number
    from google.cloud import resourcemanager_v3
    
    # Get project number
    client = resourcemanager_v3.ProjectsClient()
    project_name = f"projects/{PROJECT_ID}"
    project = client.get_project(name=project_name)
    project_number = project.name.split('/')[-1]
    
    # Construct connection name with project number
    connection_name_with_number = f"projects/{project_number}/locations/{DATASET_LOCATION}/connections/{CONNECTION_ID}"
    
    # Try to delete using gcloud command (more reliable for connections)
    import subprocess
    result = subprocess.run(
        ["gcloud", "bigquery", "connections", "delete", CONNECTION_ID,
         f"--location={DATASET_LOCATION}",
         f"--project={PROJECT_ID}",
         "--quiet"],
        capture_output=True,
        text=True
    )
    
    if result.returncode == 0:
        print("=" * 60)
        print("✅ BigQuery Connection deleted successfully!")
        print("=" * 60)
        print(f"   Connection: {CONNECTION_ID}")
        print(f"   Location: {DATASET_LOCATION}")
    elif "not found" in result.stderr.lower() or "does not exist" in result.stderr.lower():
        print("⚠️  Connection not found (already deleted or never created)")
    else:
        print(f"⚠️  Error deleting connection: {result.stderr}")
        print(f"   You can manually delete it with:")
        print(f"   gcloud bigquery connections delete {CONNECTION_ID} --location={DATASET_LOCATION} --project={PROJECT_ID}")
        
except Exception as e:
    error_msg = str(e)
    
    if "not found" in error_msg.lower() or "does not exist" in error_msg.lower():
        print("⚠️  Connection not found (already deleted or never created)")
    else:
        print(f"⚠️  Error deleting connection: {error_msg}")
        print()
        print(f"   You can manually delete it with:")
        print(f"   gcloud bigquery connections delete {CONNECTION_ID} --location={DATASET_LOCATION} --project={PROJECT_ID}")

## Section 8: Delete Cloud Storage Bucket

Delete the GCS bucket created by `upload_census_to_gcs.ipynb` and all census files.

**⚠️  WARNING:** This will permanently delete all census CSV files in the bucket.

In [ ]:
# Delete GCS bucket and all contents
print("🗑️  Deleting Cloud Storage Bucket...")
print()

print(f"⚠️  About to delete bucket: {BUCKET_NAME}")
print()

try:
    # Get bucket
    bucket = storage_client.bucket(BUCKET_NAME)
    
    if bucket.exists():
        # List all blobs in the bucket
        blobs = list(bucket.list_blobs())
        blob_count = len(blobs)
        
        print(f"   Found bucket with {blob_count} files")
        print()
        
        # Delete all blobs first
        if blob_count > 0:
            print(f"   Deleting {blob_count} files...")
            for i, blob in enumerate(blobs):
                blob.delete()
                if (i + 1) % 10 == 0 or (i + 1) == blob_count:
                    print(f"      Deleted {i + 1}/{blob_count} files", end='\r')
            print()
            print(f"   ✅ All {blob_count} files deleted")
        
        # Delete the bucket itself
        bucket.delete()
        
        print()
        print("=" * 60)
        print("✅ Cloud Storage bucket deleted successfully!")
        print("=" * 60)
        print(f"   Bucket: {BUCKET_NAME}")
        print(f"   Files deleted: {blob_count}")
    else:
        print("⚠️  Bucket not found (already deleted or never created)")
        
except Exception as e:
    error_msg = str(e)
    
    if "404" in error_msg or "not found" in error_msg.lower():
        print("⚠️  Bucket not found (already deleted or never created)")
    else:
        print(f"❌ Error deleting bucket: {error_msg}")
        raise

## Section 9: Delete CloudSQL PostgreSQL Instance

Delete the CloudSQL instance created by `setup_cloudsql_census.ipynb`.

**⚠️  WARNING:** This will permanently delete the PostgreSQL instance and all databases.

**Note:** CloudSQL deletion takes approximately 5-10 minutes to complete.

In [ ]:
# Check if CloudSQL instance exists
print("🔍 Checking if CloudSQL instance exists...")
print()

cloudsql_url = f"https://sqladmin.googleapis.com/v1/projects/{PROJECT_ID}/instances/{CLOUDSQL_INSTANCE_NAME}"
headers = {
    "Authorization": f"Bearer {credentials.token}",
    "Content-Type": "application/json"
}

try:
    response = requests.get(cloudsql_url, headers=headers)
    
    if response.status_code == 200:
        instance_info = response.json()
        print(f"✅ CloudSQL instance '{CLOUDSQL_INSTANCE_NAME}' found")
        print(f"   State: {instance_info.get('state', 'UNKNOWN')}")
        print(f"   Version: {instance_info.get('databaseVersion', 'UNKNOWN')}")
        CLOUDSQL_EXISTS = True
    elif response.status_code == 404:
        print(f"⚠️  CloudSQL instance '{CLOUDSQL_INSTANCE_NAME}' not found (already deleted or never created)")
        CLOUDSQL_EXISTS = False
    else:
        print(f"⚠️  Error checking instance: {response.status_code}")
        print(f"   Response: {response.text[:200]}")
        CLOUDSQL_EXISTS = False
        
except Exception as e:
    print(f"⚠️  Error checking CloudSQL instance: {e}")
    CLOUDSQL_EXISTS = False

In [ ]:
# Delete CloudSQL instance
print()
print("🗑️  Deleting CloudSQL PostgreSQL instance...")
print()

if not CLOUDSQL_EXISTS:
    print("⚠️  Instance not found - skipping deletion")
else:
    print(f"⚠️  About to delete instance: {CLOUDSQL_INSTANCE_NAME}")
    print("⏱️  This operation takes approximately 5-10 minutes")
    print()
    
    delete_url = f"https://sqladmin.googleapis.com/v1/projects/{PROJECT_ID}/instances/{CLOUDSQL_INSTANCE_NAME}"
    
    try:
        response = requests.delete(delete_url, headers=headers)
        
        if response.status_code in [200, 202]:
            operation = response.json()
            operation_name = operation.get('name')
            print(f"✅ Instance deletion started")
            print(f"   Operation: {operation_name}")
            print()
            print("⏳ Waiting for deletion to complete...")
            
            import time
            max_wait_time = 900  # 15 minutes
            start_time = time.time()
            last_status = None
            
            while time.time() - start_time < max_wait_time:
                # Refresh credentials if needed
                credentials.refresh(Request())
                headers["Authorization"] = f"Bearer {credentials.token}"
                
                op_url = f"https://sqladmin.googleapis.com/v1/{operation_name}"
                op_response = requests.get(op_url, headers=headers)
                
                if op_response.status_code == 200:
                    op_data = op_response.json()
                    status = op_data.get('status')
                    
                    if status == 'DONE':
                        if 'error' in op_data:
                            print(f"\n❌ Instance deletion failed: {op_data['error']}")
                            break
                        else:
                            elapsed = int(time.time() - start_time)
                            minutes = elapsed // 60
                            seconds = elapsed % 60
                            print(f"\n✅ CloudSQL instance deleted successfully! (took {minutes}m {seconds}s)")
                            break
                    else:
                        elapsed = int(time.time() - start_time)
                        minutes = elapsed // 60
                        seconds = elapsed % 60
                        if status != last_status:
                            print(f"\n   Status: {status} - Elapsed: {minutes}m {seconds}s")
                            last_status = status
                        else:
                            print(f"   Status: {status} - Elapsed: {minutes}m {seconds}s", end='\r')
                
                time.sleep(15)  # Check every 15 seconds
            
            print()
            print("=" * 60)
            print("✅ CloudSQL instance deletion completed!")
            print("=" * 60)
            print(f"   Instance: {CLOUDSQL_INSTANCE_NAME}")
            
        elif response.status_code == 404:
            print("⚠️  Instance not found (already deleted)")
        else:
            print(f"❌ Failed to delete instance: {response.status_code}")
            print(f"   Response: {response.text[:200]}")
            
    except Exception as e:
        print(f"❌ Error deleting CloudSQL instance: {e}")

## Section 10: Delete Dataplex Custom Entry, Entry Type & Entry Group

Delete the custom Dataplex catalog resources created by `write_firestore_metadata_to_dataplex.ipynb`:
- The CloudSQL entry (with its applied `data-governance-public` aspect)
- The custom entry type: `cloudsql-table`
- The custom entry group: `cloudsql-entries`

**Note:** The entry must be deleted before deleting the entry group and entry type.

In [ ]:
# Delete Dataplex custom entry
print("🗑️  Deleting Dataplex custom entry...")
print()

catalog_parent = f"projects/{PROJECT_ID}/locations/{CATALOG_LOCATION}"
entry_name = f"{catalog_parent}/entryGroups/{ENTRY_GROUP_ID}/entries/{CLOUDSQL_ENTRY_ID}"

print(f"   Entry: {CLOUDSQL_ENTRY_ID}")

try:
    catalog_client.delete_entry(name=entry_name)
    print(f"   ✅ Entry deleted: {CLOUDSQL_ENTRY_ID}")
except Exception as e:
    error_msg = str(e)
    if "404" in error_msg or "not found" in error_msg.lower():
        print(f"   ⚠️  Entry not found (already deleted or never created)")
    else:
        print(f"   ❌ Error deleting entry: {error_msg[:120]}")

print()

# Delete Dataplex custom entry type
print("🗑️  Deleting Dataplex custom entry type...")
print()

entry_type_name = f"{catalog_parent}/entryTypes/{ENTRY_TYPE_ID}"
print(f"   Entry Type: {ENTRY_TYPE_ID}")

try:
    operation = catalog_client.delete_entry_type(name=entry_type_name)
    operation.result(timeout=300)
    print(f"   ✅ Entry type deleted: {ENTRY_TYPE_ID}")
except Exception as e:
    error_msg = str(e)
    if "404" in error_msg or "not found" in error_msg.lower():
        print(f"   ⚠️  Entry type not found (already deleted or never created)")
    else:
        print(f"   ❌ Error deleting entry type: {error_msg[:120]}")

print()

# Delete Dataplex custom entry group
print("🗑️  Deleting Dataplex custom entry group...")
print()

entry_group_name = f"{catalog_parent}/entryGroups/{ENTRY_GROUP_ID}"
print(f"   Entry Group: {ENTRY_GROUP_ID}")

try:
    operation = catalog_client.delete_entry_group(name=entry_group_name)
    operation.result(timeout=300)
    print(f"   ✅ Entry group deleted: {ENTRY_GROUP_ID}")
except Exception as e:
    error_msg = str(e)
    if "404" in error_msg or "not found" in error_msg.lower():
        print(f"   ⚠️  Entry group not found (already deleted or never created)")
    else:
        print(f"   ❌ Error deleting entry group: {error_msg[:120]}")

print()
print("=" * 60)
print("✅ Dataplex custom catalog resources cleanup completed!")
print("=" * 60)

## Section 11: Delete Firestore Data

Delete the Firestore data created by `setup_firestore.ipynb` and `write_firestore_metadata_to_dataplex.ipynb`:
- All documents in the `datasets` collection (including `tables` subcollections)
- The Firestore database itself

**Note:** Firestore documents in subcollections must be deleted before the parent documents.

In [ ]:
# Delete all documents in the Firestore 'datasets' collection and subcollections
print("🗑️  Deleting Firestore data...")
print()

def delete_collection(collection_ref, batch_size=100):
    """Recursively delete all documents in a collection."""
    deleted_count = 0
    docs = list(collection_ref.limit(batch_size).stream())

    while docs:
        for doc in docs:
            # Recursively delete subcollections
            for subcol in doc.reference.collections():
                deleted_count += delete_collection(subcol, batch_size)
            doc.reference.delete()
            deleted_count += 1
        docs = list(collection_ref.limit(batch_size).stream())

    return deleted_count

try:
    datasets_ref = firestore_client.collection(FIRESTORE_DATASETS_COLLECTION)
    
    # Check if the collection has any documents
    sample = list(datasets_ref.limit(1).stream())
    
    if not sample:
        print(f"⚠️  Collection '{FIRESTORE_DATASETS_COLLECTION}' is empty or does not exist - skipping")
        firestore_deleted_count = 0
    else:
        print(f"   Deleting all documents in '{FIRESTORE_DATASETS_COLLECTION}' collection...")
        firestore_deleted_count = delete_collection(datasets_ref)
        print(f"   ✅ Deleted {firestore_deleted_count} document(s) from Firestore")

except Exception as e:
    error_msg = str(e)
    if "not found" in error_msg.lower() or "404" in error_msg:
        print(f"⚠️  Firestore database not found (already deleted or never created)")
        firestore_deleted_count = 0
    else:
        print(f"❌ Error deleting Firestore data: {error_msg[:200]}")
        firestore_deleted_count = 0

print()

# Delete the Firestore database using gcloud CLI
print("🗑️  Deleting Firestore database...")
print()
print(f"   Database: {FIRESTORE_DATABASE_ID}")

import subprocess
result = subprocess.run(
    ["gcloud", "firestore", "databases", "delete", FIRESTORE_DATABASE_ID,
     f"--project={PROJECT_ID}",
     "--quiet"],
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print(f"   ✅ Firestore database '{FIRESTORE_DATABASE_ID}' deleted")
elif "not found" in result.stderr.lower() or "does not exist" in result.stderr.lower():
    print(f"   ⚠️  Database not found (already deleted or never created)")
else:
    print(f"   ⚠️  Could not delete database via CLI: {result.stderr[:200]}")
    print(f"   You can delete it manually at:")
    print(f"   https://console.cloud.google.com/firestore/databases?project={PROJECT_ID}")

print()
print("=" * 60)
print("✅ Firestore cleanup completed!")
print("=" * 60)

## Section 12: Cleanup Summary

Review the cleanup results.

In [ ]:
# Summary
print()
print("=" * 70)
print("🎉 CLEANUP COMPLETED!")
print("=" * 70)
print()
print("📊 Summary of deleted resources:")
print()
print("   1. BigQuery Datasets:")
print(f"      - {DATASET_ID} (with {total_tables if 'total_tables' in locals() else 0} tables)")
print(f"      - {CENSUS_UK_DATASET_ID}")
print()
print("   2. BigQuery Connections:")
print(f"      - {CONNECTION_ID}")
print()
print("   3. Cloud Storage:")
print(f"      - Bucket: {BUCKET_NAME}")
print(f"      - All census CSV files")
print()
print("   4. CloudSQL:")
print(f"      - Instance: {CLOUDSQL_INSTANCE_NAME}")
print(f"      - Database: census_data")
print(f"      - Table: census_residence_type")
print()
print("   5. Dataplex Aspect Types:")
print("      - census-survey-metadata")
print("      - data-governance-public")
print()
print("   6. Dataplex Aspects:")
print(f"      - Removed from {total_tables if 'total_tables' in locals() else 0} table entries")
print()
print("   7. Dataplex Glossary:")
print(f"      - Glossary: {GLOSSARY_ID}")
print(f"      - Categories: {len(categories) if 'categories' in locals() else 0}")
print(f"      - Terms: {len(terms) if 'terms' in locals() else 0}")
print()
print("   8. Dataplex Custom Catalog Resources:")
print(f"      - Entry: {CLOUDSQL_ENTRY_ID}")
print(f"      - Entry Type: {ENTRY_TYPE_ID}")
print(f"      - Entry Group: {ENTRY_GROUP_ID}")
print()
print("   9. Firestore:")
print(f"      - Database: {FIRESTORE_DATABASE_ID}")
print(f"      - Documents deleted: {firestore_deleted_count if 'firestore_deleted_count' in locals() else 0}")
print()
print("✅ All demo resources have been cleaned up!")
print()
print("🔗 Verify in Console:")
print(f"   BigQuery Datasets: https://console.cloud.google.com/bigquery?project={PROJECT_ID}")
print(f"   BigQuery Connections: https://console.cloud.google.com/bigquery/connections?project={PROJECT_ID}")
print(f"   Cloud Storage: https://console.cloud.google.com/storage/browser?project={PROJECT_ID}")
print(f"   CloudSQL Instances: https://console.cloud.google.com/sql/instances?project={PROJECT_ID}")
print(f"   Dataplex Aspect Types: https://console.cloud.google.com/dataplex/govern/aspect-types?project={PROJECT_ID}")
print(f"   Dataplex Glossaries: https://console.cloud.google.com/dataplex/dp-glossaries?project={PROJECT_ID}")
print(f"   Dataplex Catalog: https://console.cloud.google.com/dataplex/search?project={PROJECT_ID}")
print(f"   Firestore Databases: https://console.cloud.google.com/firestore/databases?project={PROJECT_ID}")

## Optional: Clean up IAM permissions

If you want to remove the IAM permissions that were granted for the demo, run these commands in your terminal:

```bash
# Remove BigQuery Admin
gcloud projects remove-iam-policy-binding johnswain-1200-20260303091357 \
  --member='user:YOUR_EMAIL' \
  --role='roles/bigquery.admin'

# Remove Cloud Storage Admin
gcloud projects remove-iam-policy-binding johnswain-1200-20260303091357 \
  --member='user:YOUR_EMAIL' \
  --role='roles/storage.admin'

# Remove CloudSQL Admin
gcloud projects remove-iam-policy-binding johnswain-1200-20260303091357 \
  --member='user:YOUR_EMAIL' \
  --role='roles/cloudsql.admin'

# Remove Dataplex Admin
gcloud projects remove-iam-policy-binding johnswain-1200-20260303091357 \
  --member='user:YOUR_EMAIL' \
  --role='roles/dataplex.admin'

# Remove Dataplex Catalog Admin
gcloud projects remove-iam-policy-binding johnswain-1200-20260303091357 \
  --member='user:YOUR_EMAIL' \
  --role='roles/dataplex.catalogAdmin'
```

Replace `YOUR_EMAIL` with your Google Cloud account email.

In [ ]:
# Disable Dataplex integration before deleting instance
print()
print("🔗 Disabling Dataplex Universal Catalog integration...")
print()

if not CLOUDSQL_EXISTS:
    print("⚠️  Instance not found - skipping Dataplex integration disable")
else:
    try:
        patch_body = {
            "settings": {
                "enableDataplexIntegration": False
            }
        }
        
        response = requests.patch(cloudsql_url, headers=headers, json=patch_body)
        
        if response.status_code in [200, 201]:
            print("✅ Dataplex integration disabled")
            operation = response.json()
            operation_name = operation.get('name')
            
            # Wait for operation to complete (quick operation, ~30 seconds)
            print("   Waiting for operation to complete...")
            max_wait = 60
            start_time = time.time()
            
            while time.time() - start_time < max_wait:
                op_url = f"https://sqladmin.googleapis.com/v1/{operation_name}"
                op_response = requests.get(op_url, headers=headers)
                
                if op_response.status_code == 200:
                    op_data = op_response.json()
                    if op_data.get('status') == 'DONE':
                        print("   ✅ Operation completed")
                        break
                
                time.sleep(5)
        else:
            print(f"⚠️  Could not disable integration: {response.status_code}")
            print("   Proceeding with instance deletion anyway...")
            
    except Exception as e:
        print(f"⚠️  Error disabling Dataplex integration: {e}")
        print("   Proceeding with instance deletion anyway...")